# Uitleg van de hele code en het dashboard

**Voor iemand die ongeveer 1 maand een data-analyse cursus heeft gevolgd**

In dit notebook leggen we stap voor stap uit:
- welke bestanden belangrijk zijn
- hoe de data wordt opgehaald en samengevoegd
- hoe de Streamlit-app is opgebouwd
- wat elke grafiek en tabel doet
- welke stukken code je als beginner echt moet begrijpen

We houden de uitleg bewust simpel. Het doel is niet om moeilijke woorden te gebruiken,
maar om te snappen **wat de code doet** en **waarom**.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, r"c:\Users\AAchr\Documents\Casus 2 phyton minor\ROC")

from data import laad_alle_data, ADVIES_TYPEN, SCHOOLJAREN

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

wijken_df, duo_df, scholen_df, bronnen = laad_alle_data()

print("Bronnen:")
for naam, bron in bronnen.items():
    print(f"- {naam}: {bron}")

print()
print("Aantal rijen:")
print("wijken_df:", len(wijken_df))
print("duo_df:", len(duo_df))
print("scholen_df:", len(scholen_df))

2026-04-06 19:10:57.140 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-04-06 19:10:57.142 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-04-06 19:10:57.143 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-04-06 19:10:57.144 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-04-06 19:10:57.145 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


...

2026-04-06 19:10:57.147 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-04-06 19:10:57.147 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-04-06 19:10:57.391 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-04-06 19:10:57.950 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Bronnen:
- CBS Kerncijfers Wijken en Buurten 2024: Live data van CBS OData API (samengevoegd naar dashboardwijken)
- DUO Schooladviezen per school: Live data van DUO Open Onderwijsdata (adviescodes samengevoegd, bijstelling niet apart beschikbaar)
- Amsterdam schoolgebouwen: Live data van Amsterdam schoolgebouwen API

Aantal rijen:
wijken_df: 22
duo_df: 2971
scholen_df: 214


## 1. Welke bestanden zijn belangrijk?

Voor dit project zijn vooral deze 2 bestanden belangrijk:

- `data.py`
  Hier wordt de data opgehaald, opgeschoond en samengevoegd.
- `app.py`
  Hier wordt de Streamlit-app gebouwd: tabs, filters, grafieken en tabellen.

Je kunt het zo zien:

- `data.py` = de **keuken**
- `app.py` = het **restaurant**

Eerst wordt in de keuken alles voorbereid. Daarna laat het restaurant het netjes aan de gebruiker zien.

In [2]:
def toon_regels(bestand, start, einde):
    """Laat een stukje code zien met regelnummers."""
    regels = Path(bestand).read_text(encoding="utf-8").splitlines()
    for i in range(start - 1, einde):
        print(f"{i+1:>4}: {regels[i]}")

## 2. Uitleg van `data.py`

Dit bestand doet 4 hoofdzaken:

1. vaste lijsten en kleuren maken
2. live data ophalen van CBS
3. live data ophalen van DUO en Amsterdam
4. alles samenvoegen tot datasets die de app kan gebruiken

Dat zijn goede beginnerstappen:
- eerst data ophalen
- dan kolommen gelijk maken
- dan datasets koppelen
- dan pas grafieken bouwen

In [3]:
print("Belangrijke start van data.py")
toon_regels("data.py", 1, 120)

Belangrijke start van data.py
   1: # data.py
   2: # Gemaakt door: Groep 2
   3: # Hier halen we alle data op die we nodig hebben voor ons dashboard.
   4: # We gebruiken drie bronnen:
   5: #   1. CBS OData API  - inkomen en achtergrond per wijk
   6: #   2. DUO Open Data  - schooladviezen per school per jaar
   7: #   3. Amsterdam API  - schoollocaties met stadsdeel info
   8: 
   9: import requests
  10: import pandas as pd
  11: import numpy as np
  12: import streamlit as st
  13: 
  14: # alle adviestypen in volgorde van laag naar hoog
  15: ADVIES_TYPEN = [
  16:     "Praktijkonderwijs",
  17:     "VMBO-BBL",
  18:     "VMBO-KBL",
  19:     "VMBO-TL",
  20:     "VMBO-TL/HAVO",
  21:     "HAVO",
  22:     "HAVO/VWO",
  23:     "VWO",
  24: ]
  25: 
  26: # kleur per adviestype (rood = laag, groen = hoog)
  27: ADVIES_KLEUREN = {
  28:     "Praktijkonderwijs": "#d73027",
  29:     "VMBO-BBL":          "#f46d43",
  30:     "VMBO-KBL":          "#fdae61",
  31:     "VMBO-TL":      

### 2.1 Vaste lijsten en kleuren

Bovenin `data.py` staan vaste dingen zoals:
- `ADVIES_TYPEN`
- `ADVIES_KLEUREN`
- `SCHOOLJAREN`

Waarom is dat handig?

Omdat je die informatie daarna op meerdere plekken opnieuw kunt gebruiken.
Bijvoorbeeld:
- in filters
- in grafieken
- in legenda's
- in berekeningen

Dat voorkomt dat je op 5 plekken steeds dezelfde lijst opnieuw moet typen.

### 2.2 CBS-data

De functie `haal_cbs_data_op()` haalt wijkgegevens op, zoals:
- inkomen
- aantal inwoners
- aandeel niet-westerse achtergrond
- opleidingsniveau
- uitkering
- WOZ-waarde

Belangrijk om te begrijpen:

De live CBS-data gebruikt **andere wijkcodes** dan jullie dashboard.
Daarom wordt de live data later omgezet naar jullie eigen wijkgroepen.
Dat is een heel normale stap in data-analyse:
de brondata past niet altijd meteen op jouw onderzoeksvraag.

In [4]:
print("CBS-functies in data.py")
toon_regels("data.py", 40, 185)

CBS-functies in data.py
  40: # De live DUO-data gebruikt numerieke adviescodes. Voor dit dashboard zetten we die
  41: # grof om naar dezelfde 8 categorieen als in de rest van de app.
  42: DUO_ADVIES_NAAR_TYPE = {
  43:     1:  "Praktijkonderwijs",
  44:     2:  "VMBO-BBL",
  45:     3:  "VMBO-BBL",
  46:     4:  "VMBO-KBL",
  47:     5:  "VMBO-KBL",
  48:     6:  "VMBO-TL",
  49:     7:  "VMBO-TL",
  50:     8:  "VMBO-TL/HAVO",
  51:     9:  "HAVO",
  52:     10: "HAVO",
  53:     11: "HAVO/VWO",
  54:     12: "VWO",
  55: }
  56: 
  57: 
  58: def koppel_cbs_code_aan_dashboard(wijk_code):
  59:     code = str(wijk_code).strip()
  60: 
  61:     if code.startswith("WK0363A"):
  62:         return "Centrum"
  63:     if code.startswith("WK0363B"):
  64:         return "Westelijk Havengebied"
  65: 
  66:     if code in ["WK0363EA", "WK0363EB", "WK0363EC", "WK0363ED", "WK0363EE"]:
  67:         return "Westerpark"
  68:     if code in ["WK0363EF", "WK0363EG", "WK0363EH", "WK0363EJ", "

### 2.3 DUO-data

De DUO-data gaat over schooladviezen.

Daar zie je per school en per jaar:
- hoeveel leerlingen een bepaald advies kregen

In jullie code worden numerieke DUO-codes vertaald naar begrijpelijke adviesgroepen, zoals:
- `VMBO-BBL`
- `VMBO-KBL`
- `VMBO-TL`
- `HAVO`
- `HAVO/VWO`
- `VWO`

Dat is belangrijk, want gebruikers willen geen code `7` of `11` zien,
maar een echte naam.

In [5]:
print("DUO-functies in data.py")
toon_regels("data.py", 150, 290)

DUO-functies in data.py
 150:                 elif "inwoners" in k and "aantal" in k:
 151:                     nieuwe_naam = "aantal_inwoners"
 152:                 elif "nietwester" in k.replace(" ", "").replace("-", "") or \
 153:                      ("buiten" in k and "europa" in k):
 154:                     nieuwe_naam = "pct_niet_westers"
 155:                 elif "bijstand" in k or ("uitkering" in k and "relatief" in k) or \
 156:                      ("bijstand" in k and "personen" in k):
 157:                     nieuwe_naam = "pct_uitkering"
 158:                 elif ("basisonderwijs" in k or "vmbo" in k or "mbo1" in k) and \
 159:                      ("laag" in k or "basis" in k or "vmbo" in k):
 160:                     nieuwe_naam = "pct_laag_opgeleid"
 161:                 elif ("hbo" in k or "wo" in k) and ("hoog" in k or "hbo" in k):
 162:                     nieuwe_naam = "pct_hoog_opgeleid"
 163:                 elif "woz" in k and "gemiddeld" in k:
 164:        

### 2.4 Amsterdam schooldata en wijkgrenzen

De Amsterdam API levert in deze app nu 2 soorten data:
- schoolinformatie
- wijkgrenzen

De schooldata gebruiken we voor:
- schoolnaam
- BRIN
- stadsdeel
- wijk

De wijkgrenzen gebruiken we voor de kaart van Amsterdam.
Daardoor worden wijken nu niet meer als losse punten getoond,
maar als echte vlakken met een rand eromheen.

Let op:
- schoollocaties worden nog steeds benaderd met wijk- of stadsdeelcentra
- de wijkkaart zelf gebruikt wel echte polygonen uit de Amsterdam API
- de hover op de wijkkaart laat nu ook aantallen per adviestype zien


In [6]:
print("Amsterdam schooldata en koppeling")
toon_regels("data.py", 290, 430)

Amsterdam schooldata en koppeling
 290:             )
 291:             df["schooljaar"] = (df["PEILJAAR"] - 1).astype(str) + "-" + df["PEILJAAR"].astype(str)
 292:             df["advies_type"] = df["ADVIES"].map(DUO_ADVIES_NAAR_TYPE)
 293:             df["aantal_leerlingen"] = pd.to_numeric(df["AANTAL_LEERLINGEN"], errors="coerce").fillna(0)
 294: 
 295:             df = df[
 296:                 df["schooljaar"].isin(SCHOOLJAREN) &
 297:                 df["advies_type"].notna() &
 298:                 (df["aantal_leerlingen"] >= 0)
 299:             ].copy()
 300: 
 301:             # Koppel live DUO aan de Amsterdam schooldata via BRIN.
 302:             df = df.merge(
 303:                 scholen_df[["brin", "school_naam", "stadsdeel", "wijk_naam"]],
 304:                 on="brin",
 305:                 how="inner"
 306:             )
 307: 
 308:             # Gebruik de bekende Amsterdam wijkcodes uit onze vaste wijktabel.
 309:             wijk_info = maak_cbs_nooddata()[["w

### 2.5 Alles samenvoegen in `laad_alle_data()`

Dit is de belangrijkste functie van `data.py`.

Daar gebeurt ongeveer dit:

1. haal CBS, DUO en Amsterdam data op
2. koppel scholen aan wijken
3. bereken percentages
4. maak een samenvatting per wijk
5. geef de datasets terug aan de app

Daarnaast wordt er ook nog een aparte functie gebruikt voor de wijkgrenzenkaart:
- `haal_amsterdam_wijkgrenzen_op()`

Het resultaat zijn 3 hoofdtabellen:

- `wijken_df`
  Alles op wijkniveau
- `duo_df`
  Alles op school x jaar x adviestype
- `scholen_df`
  Schoolnamen en kaartlocaties

En daarnaast is er nog:
- `wijk_geojson`
  De echte wijkgrenzen voor de kaart


In [7]:
print("Belangrijkste samenvoegfunctie")
toon_regels("data.py", 430, 520)

print("\nVoorbeeld van wijken_df")
display(wijken_df[["wijk_naam", "stadsdeel", "gem_inkomen", "pct_hoog_advies", "pct_laag_advies"]].head())

print("Voorbeeld van duo_df")
display(duo_df[["school_naam", "wijk_naam", "schooljaar", "advies_type", "aantal_leerlingen", "pct"]].head())

print("Voorbeeld van scholen_df")
display(scholen_df.head())

Belangrijkste samenvoegfunctie
 430:         accommodaties_records = accommodaties_resp.json().get("_embedded", {}).get("accommodatie", [])
 431: 
 432:         if len(instellingen_records) == 0 or len(accommodaties_records) == 0:
 433:             raise ValueError("Amsterdam API gaf geen schoolrecords terug")
 434: 
 435:         instellingen_df = pd.DataFrame(instellingen_records)
 436:         accommodaties_df = pd.DataFrame(accommodaties_records)
 437: 
 438:         scholen_df = instellingen_df.merge(
 439:             accommodaties_df[["instellingId", "stadsdeel", "wijk", "adresStraat", "archief", "mIsActief"]].rename(columns={
 440:                 "archief": "archief_accommodatie",
 441:                 "mIsActief": "actief_accommodatie"
 442:             }),
 443:             on="instellingId",
 444:             how="left"
 445:         )
 446: 
 447:         scholen_df = scholen_df[
 448:             scholen_df["brinNummerDuo"].notna() &
 449:             scholen_df["naam"].n

,wijk_naam,stadsdeel,gem_inkomen,pct_hoog_advies,pct_laag_advies
0,Centrum,Centrum,58.230000,94.8,NaN
1,Oud-West / De Baarsjes,West,54.133333,70.3,4.3
2,Westerpark,West,51.275000,73.8,NaN
3,Geuzenveld-Slotermeer,Nieuw-West,34.375000,49.8,13.9
4,Osdorp,Nieuw-West,38.175000,48.9,8.6


Voorbeeld van duo_df


,school_naam,wijk_naam,schooljaar,advies_type,aantal_leerlingen,pct
0,Prof. Dumont,Gaasperdam / Driemond,2018-2019,VMBO-BBL,9,4.7
1,Prof. Dumont,Gaasperdam / Driemond,2019-2020,VMBO-BBL,20,9.0
2,Prof. Dumont,Gaasperdam / Driemond,2020-2021,VMBO-BBL,10,6.0
3,Prof. Dumont,Gaasperdam / Driemond,2021-2022,Praktijkonderwijs,6,2.4
4,Prof. Dumont,Gaasperdam / Driemond,2021-2022,VMBO-BBL,21,8.6


Voorbeeld van scholen_df


,brin,school_naam,stadsdeel,wijk_naam,adresStraat,lat,lon
0,20XC00,Kans,Nieuw-West,Geuzenveld-Slotermeer,Thomas van Aquinostraat,52.3966,4.7950
1,21AG00,Globe,Nieuw-West,Osdorp,Evertsweertplantsoen,52.3696,4.8070
2,24BM00,Vlaamse Reus,Nieuw-West,Slotervaart,Hechtelstraat,52.3654,4.8185
3,25KC00,Horizon,Nieuw-West,"Aker, Sloten en Nieuw Sloten",Pieter Calandlaan,52.3402,4.8055
4,05LX00,Veerkracht,Nieuw-West,Geuzenveld-Slotermeer,Slotermeerlaan,52.3822,4.8010


## 3. Uitleg van `app.py`

`app.py` is de presentatie-laag.

Daar gebeurt dit:

1. pagina-instellingen zetten
2. data inladen
3. filters in de sidebar maken
4. tabs maken
5. grafieken en tabellen tonen

Dit is precies hoe veel dashboards zijn opgebouwd.

In [8]:
print("Start van app.py")
toon_regels("app.py", 1, 120)

Start van app.py
   1: # app.py
   2: # Doorstroomtoets & Kansengelijkheid in Amsterdam
   3: # Gemaakt door: Groep 2
   4: #
   5: # We onderzoeken of de doorstroomtoets (ingevoerd in 2023-2024) helpt
   6: # om de kansenongelijkheid in Amsterdam te verkleinen.
   7: # Daarnaast kijken we naar kansengelijkheid in schooladviezen tussen Amsterdamse wijken.
   8: 
   9: import streamlit as st
  10: import pandas as pd
  11: import numpy as np
  12: import plotly.express as px
  13: import plotly.graph_objects as go
  14: from scipy import stats
  15: from sklearn.linear_model import LinearRegression
  16: from sklearn.preprocessing import StandardScaler
  17: 
  18: from data import laad_alle_data, haal_amsterdam_wijkgrenzen_op, maak_duo_nooddata, ADVIES_TYPEN, ADVIES_KLEUREN, SCHOOLJAREN
  19: 
  20: # ---------------------------------------------------------------
  21: # Pagina instellingen
  22: # ---------------------------------------------------------------
  23: st.set_page_confi

### 3.1 De sidebar met filters

In de sidebar kiest de gebruiker:
- stadsdeel
- wijk
- schooljaren
- adviestypen

Daarna worden de datasets gefilterd.

Dit is een heel belangrijk idee in dashboards:

- **eerst filteren**
- **daarna pas tekenen**

Zo veranderen alle grafieken automatisch mee.

In [9]:
print("Filters en tabdefinitie")
toon_regels("app.py", 120, 210)

Filters en tabdefinitie
 120: 
 121:     st.markdown("---")
 122: 
 123:     # --- Filter 3: Schooladvies typen (checkboxen met conflict-logica) ---
 124:     st.subheader("🎓 Adviestypen")
 125:     st.caption("Vink aan welke adviezen je wilt zien")
 126: 
 127:     gekozen_adviezen = []
 128:     for advies in ADVIES_TYPEN:
 129:         kleur = ADVIES_KLEUREN[advies]
 130:         aangevinkt = st.checkbox(
 131:             advies,
 132:             value=True,
 133:             key=f"cb_{advies}"
 134:         )
 135:         if aangevinkt:
 136:             gekozen_adviezen.append(advies)
 137: 
 138:     # als de gebruiker niets aanvinkt, geef een melding
 139:     if len(gekozen_adviezen) == 0:
 140:         st.error("⚠️ Vink minimaal één adviestype aan om resultaten te zien.")
 141:         gekozen_adviezen = ADVIES_TYPEN  # terugvallen op alles
 142: 
 143:     # knop om alles te selecteren / deselecteren
 144:     col1, col2 = st.columns(2)
 145:     with col1:
 146:         i

## 4. Uitleg per tab, grafiek en tabel

Hieronder leggen we per onderdeel uit:
- wat je ziet
- welke data gebruikt wordt
- hoe je de grafiek moet lezen

### Tab 1 - Introductie

Deze tab bevat vooral tekst en 4 kerncijfers:
- gemiddelde % hoog advies
- hoogste wijk
- laagste wijk
- verschil tussen wijken

Waarom is dit handig?

Omdat een gebruiker meteen het grote verhaal ziet voordat hij naar detailgrafieken gaat kijken.

In [10]:
toon_regels("app.py", 210, 275)

 210: 
 211:     Vroeger heette het de **eindtoets** (CITO). Vanaf het schooljaar **2023-2024** heet het
 212:     de **doorstroomtoets**. Het grote verschil:
 213: 
 214:     - Bij de *eindtoets* kon de score het schooladvies omhoog **of omlaag** bijstellen.
 215:     - Bij de *doorstroomtoets* kan het advies alleen nog maar **omhoog** worden bijgesteld.
 216: 
 217:     Het doel is om kansenongelijkheid te verkleinen: kinderen uit armere wijken werden
 218:     vroeger vaker te laag ingeschat, ook al scoorden ze goed op de toets.
 219: 
 220:     ### Onze onderzoeksvraag
 221: 
 222:     > **Helpt de doorstroomtoets kinderen in achterstandswijken van Amsterdam?**
 223: 
 224:     ### Wat je in dit dashboard vindt
 225: 
 226:     | Tab | Inhoud |
 227:     |-----|--------|
 228:     | 🗺️ Kaart | Overzicht van Amsterdam met schooladviezen per wijk |
 229:     | 📊 Schooladviezen | Alle adviestypen (PrO t/m VWO) per wijk vergeleken |
 230:     | 🎓 Doorstroomtoets | Voor en na vergelijki

### Tab 2 - Kaart Amsterdam

Hier staan 2 kaartdelen:

1. **Wijkkaart**
   Elke wijk wordt als echt vlak getekend en krijgt een kleur op basis van `% hoog advies`.

2. **Schoollocaties**
   Scholen worden als punten op de kaart gezet.

Hoe lees je dit?

- groener = meer hoge adviezen
- roder = minder hoge adviezen
- in de hover van een wijk zie je nu ook de aantallen per adviestype
- de wijkrand laat de vorm van de wijk zien

Dit is beter dan de oude puntkaart, omdat je nu duidelijker ziet:
- welke wijk waar ligt
- hoe groot de wijk is
- welke adviezen in die wijk voorkomen


In [11]:
toon_regels("app.py", 275, 345)

 275: 
 276:     if (
 277:         "pct_hoog_advies" in wijken_gefilterd.columns and
 278:         wijk_geojson is not None and
 279:         not wijkgrenzen_df.empty
 280:     ):
 281:         kaart_basis = wijken_gefilterd.dropna(subset=["pct_hoog_advies"]).copy()
 282: 
 283:         advies_aantallen = (
 284:             duo_gefilterd
 285:             .groupby(["wijk_naam", "advies_type"], as_index=False)
 286:             .agg(aantal=("aantal_leerlingen", "sum"))
 287:         )
 288:         advies_aantallen = (
 289:             advies_aantallen
 290:             .pivot(index="wijk_naam", columns="advies_type", values="aantal")
 291:             .fillna(0)
 292:             .reset_index()
 293:         )
 294:         advies_aantallen.columns.name = None
 295: 
 296:         advies_kolommen = []
 297:         hernoem_adviezen = {}
 298:         for advies in ADVIES_TYPEN:
 299:             veilige_kolom = (
 300:                 "aantal_" + advies.lower()
 301:                

### Tab 3 - Schooladviezen

Deze tab heeft 3 onderdelen:

1. **Gestapelde staafgrafiek**
   Laat per wijk zien hoe de adviesverdeling is opgebouwd.

2. **Scatterplot inkomen vs hoog advies**
   Laat zien of rijkere wijken ook meer hoge adviezen hebben.

3. **Heatmap**
   Laat per wijk en per advies_type het percentage zien.

Waarom is dit nuttig?

Omdat je hetzelfde onderwerp op 3 manieren bekijkt:
- verdeling
- verband
- overzicht

In [12]:
toon_regels("app.py", 345, 455)

 345:         hoverregels = [
 346:             "<b>%{hovertext}</b>",
 347:             "Stadsdeel: %{customdata[0]}",
 348:             "Gem. inkomen: %{customdata[1]:.1f}",
 349:             "% hoog advies: %{customdata[2]:.1f}%",
 350:             "% laag advies: %{customdata[3]:.1f}%",
 351:             "<br><b>Aantallen per adviestype</b>",
 352:         ]
 353:         for index, (advies, _) in enumerate(advies_kolommen, start=4):
 354:             hoverregels.append(f"{advies}: %{{customdata[{index}]:.0f}}")
 355:         fig_kaart.update_traces(hovertemplate="<br>".join(hoverregels) + "<extra></extra>")
 356: 
 357:         fig_kaart.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
 358:         st.plotly_chart(fig_kaart, width="stretch")
 359:         st.caption(f"Wijkgrenzen uit: {wijkgrenzen_bron}. Hover toont aantallen adviezen in de gekozen periode.")
 360:     else:
 361:         st.info("Kaart niet beschikbaar voor de huidige selectie.")
 362: 
 363:     st.markd

### Tab 4 - Doorstroomtoets

Deze tab vergelijkt:
- `2022-2023` = oude situatie
- `2023-2024` = eerste jaar doorstroomtoets

Grafieken:

1. **Voor/na staafgrafiek**
   Vergelijkt adviesverdelingen tussen de 2 jaren.

2. **Verschilgrafiek**
   Laat direct zien welke adviezen stegen of daalden.

3. **Bijgesteld advies per wijk**
   Laat zien in welke wijken het meeste omhoog is bijgesteld.

Belangrijk in de huidige code:
- de app probeert eerst live DUO-data te gebruiken
- als die bron geen bruikbare bijstellingsdata bevat, gebruikt deze tab automatisch de projectdataset
- dat gebeurt alleen voor deze tab, omdat juist hier het effect van de doorstroomtoets centraal staat

Als beginner moet je hier vooral snappen:
- je vergelijkt twee jaren
- je berekent verschillen
- je kiest soms bewust een betere databron voor precies die analyse die je wilt doen


In [13]:
toon_regels("app.py", 455, 560)

 455:         st.subheader("📉 Vergelijking: inkomen vs. schooladvies")
 456:         st.markdown("""
 457:         De grafiek hieronder laat zien of er een verband is tussen het **gemiddelde inkomen**
 458:         in een wijk en het percentage **hoge adviezen** (HAVO, HAVO/VWO, VWO).
 459:         """)
 460: 
 461:         scatter_df = wijken_gefilterd.dropna(subset=["gem_inkomen", "pct_hoog_advies"])
 462:         if len(scatter_df) >= 3:
 463:             # Pearson correlatie berekenen
 464:             r, p = stats.pearsonr(scatter_df["gem_inkomen"], scatter_df["pct_hoog_advies"])
 465: 
 466:             # size en hover_data alleen als pct_niet_westers beschikbaar is
 467:             scatter_kwargs = dict(
 468:                 x="gem_inkomen",
 469:                 y="pct_hoog_advies",
 470:                 color="stadsdeel",
 471:                 hover_name="wijk_naam",
 472:                 hover_data={"gem_inkomen": True, "pct_hoog_advies": True},
 473:                 trendl

### Tab 5 - Voorspelling

Hier wordt een simpel regressiemodel gebruikt.

Het model probeert `% hoog advies` te voorspellen op basis van:
- `gem_inkomen`
- `pct_niet_westers`
- `pct_uitkering`
- `pct_laag_opgeleid`

Belangrijke woorden:

- **voorspeld** = wat het model denkt
- **werkelijk** = wat echt in de data staat
- **residu** = verschil tussen die twee

Waarom is dat nuttig?

Omdat je zo kunt zien welke wijken beter of slechter scoren dan je op basis van hun achtergrond zou verwachten.

In [14]:
toon_regels("app.py", 560, 705)

 560:             .agg(aantal=("aantal_leerlingen", "sum"))
 561:         )
 562:         na_totaal = na_gem["aantal"].sum()
 563:         na_gem["2023-2024"] = (na_gem["aantal"] / na_totaal * 100).round(1)
 564:         na_gem = na_gem[["advies_type", "2023-2024"]]
 565: 
 566:         vergelijk = voor_gem.merge(na_gem, on="advies_type")
 567:         vergelijk["verandering"] = (vergelijk["2023-2024"] - vergelijk["2022-2023"]).round(2)
 568:         vergelijk = vergelijk[vergelijk["advies_type"].isin(gekozen_adviezen)]
 569: 
 570:         col1, col2 = st.columns(2)
 571: 
 572:         with col1:
 573:             fig_voorna = go.Figure()
 574:             fig_voorna.add_trace(go.Bar(
 575:                 name="2022-2023 (eindtoets)",
 576:                 x=vergelijk["advies_type"],
 577:                 y=vergelijk["2022-2023"],
 578:                 marker_color="#90a4ae"
 579:             ))
 580:             fig_voorna.add_trace(go.Bar(
 581:                 name="2023-2024 (do

### Tab 6 - Data & Methoden

Dit is de uitleg-tab van het dashboard zelf.

Hier laat de app zien:
- welke databronnen zijn gebruikt
- hoe de APIs werken
- hoe datasets zijn gekoppeld
- ruwe tabellen

Waarom is dit belangrijk?

Omdat een dashboard niet alleen mooie grafieken moet tonen,
maar ook eerlijk moet laten zien **waar de cijfers vandaan komen**.

In de huidige versie is dat extra belangrijk, omdat:
- CBS live wordt gebruikt
- DUO live wordt gebruikt voor de algemene schooladviezen
- Amsterdam live wordt gebruikt voor scholen en wijkgrenzen
- de doorstroomtoets-tab zo nodig terugvalt op de projectdataset voor bijstellingen


In [15]:
toon_regels("app.py", 705, 860)

 705: 
 706:         model_data = model_data.copy()
 707:         model_data["voorspeld"] = y_pred.round(1)
 708:         model_data["werkelijk"] = y.round(1)
 709:         model_data["residu"]    = residu.round(1)
 710:         model_data["prestatie"] = model_data["residu"].apply(
 711:             lambda r: "Beter dan verwacht" if r > 3
 712:             else "Slechter dan verwacht" if r < -3
 713:             else "Zoals verwacht"
 714:         )
 715: 
 716:         st.markdown(f"**Model R² = {r2:.3f}** — het model verklaart {r2*100:.1f}% van de variatie in schooladvies.")
 717: 
 718:         col1, col2 = st.columns(2)
 719:         with col1:
 720:             fig_model = px.scatter(
 721:                 model_data,
 722:                 x="voorspeld",
 723:                 y="werkelijk",
 724:                 color="prestatie",
 725:                 text="wijk_naam",
 726:                 color_discrete_map={
 727:                     "Beter dan verwacht":    "#1a9850",
 728:  

## 5. Uitleg van de tabellen in de app

Onderaan de app staan ook tabellen met ruwe data.

Dat is goed, want:
- grafieken geven een samenvatting
- tabellen laten de echte waarden zien

Voor beginners is dit een mooie regel:

**Gebruik een grafiek om een verhaal te vertellen, en een tabel om te controleren of het verhaal klopt.**

In [16]:
print("Voorbeeld van een wijktabel")
display(wijken_df[["wijk_naam", "stadsdeel", "gem_inkomen", "pct_hoog_advies", "pct_laag_advies"]].head(10))

print("Voorbeeld van een schooladviestabel")
display(duo_df[["school_naam", "wijk_naam", "schooljaar", "advies_type", "aantal_leerlingen", "pct"]].head(10))

Voorbeeld van een wijktabel


,wijk_naam,stadsdeel,gem_inkomen,pct_hoog_advies,pct_laag_advies
0,Centrum,Centrum,58.230000,94.8,NaN
1,Oud-West / De Baarsjes,West,54.133333,70.3,4.3
2,Westerpark,West,51.275000,73.8,NaN
3,Geuzenveld-Slotermeer,Nieuw-West,34.375000,49.8,13.9
4,Osdorp,Nieuw-West,38.175000,48.9,8.6
5,Slotervaart,Nieuw-West,40.166667,54.0,11.9
6,Bos en Lommer,West,48.475000,NaN,NaN
7,Oud-Zuid,Zuid,83.025000,89.7,1.4
8,De Pijp / Rivierenbuurt,Zuid,51.800000,80.0,NaN
9,Buitenveldert / Zuidas,Zuid,66.725000,80.5,NaN


Voorbeeld van een schooladviestabel


,school_naam,wijk_naam,schooljaar,advies_type,aantal_leerlingen,pct
0,Prof. Dumont,Gaasperdam / Driemond,2018-2019,VMBO-BBL,9,4.7
1,Prof. Dumont,Gaasperdam / Driemond,2019-2020,VMBO-BBL,20,9.0
2,Prof. Dumont,Gaasperdam / Driemond,2020-2021,VMBO-BBL,10,6.0
3,Prof. Dumont,Gaasperdam / Driemond,2021-2022,Praktijkonderwijs,6,2.4
4,Prof. Dumont,Gaasperdam / Driemond,2021-2022,VMBO-BBL,21,8.6
5,Prof. Dumont,Gaasperdam / Driemond,2022-2023,Praktijkonderwijs,6,3.0
6,Prof. Dumont,Gaasperdam / Driemond,2022-2023,VMBO-BBL,14,7.1
7,Prof. Dumont,Gaasperdam / Driemond,2023-2024,VMBO-BBL,14,8.5
8,Tobiasschool,Oud-Zuid,2018-2019,VMBO-KBL,6,1.0
9,Tobiasschool,Oud-Zuid,2019-2020,VMBO-BBL,5,0.9


## 6. Wat moet je als beginner vooral onthouden?

Als je pas kort met data-analyse bezig bent, probeer dan vooral dit te onthouden:

1. Data ophalen is vaak maar het begin.
   Je moet data daarna bijna altijd nog opschonen en koppelen.

2. Een dashboard bestaat uit kleine stappen.
   Eerst data inladen, dan filteren, dan berekenen, dan tekenen.

3. Een grafiek is een vraag in beeldvorm.
   Vraag jezelf altijd af: wat wil ik hier laten zien?

4. Niet alles hoeft meteen perfect of slim.
   Duidelijke, simpele code is vaak beter dan ingewikkelde code.

5. Tabellen blijven belangrijk.
   Daarmee kun je controleren of je grafiek logisch is.

Als je deze 5 dingen begrijpt, dan begrijp je de basis van dit project al goed.